# RoadSense: Robust Traffic Sign Recognition
## Phase 2: Baseline CNN from Scratch
**Group 4 | Atria University**  
- **Primary Owner:** Raju (Scratch CNN Design, Training Loop, Evaluation, Confusion Matrix)  
- **Team:** Aseem (Leader), Sohail, Manoj  

---
### Objectives of Phase 2:
1. **Architecture Design & Justification:** Build a custom CNN tailored for $32 \times 32$ RGB traffic sign images from scratch without pre-trained backbones.
2. **Handle Class Imbalance:** Apply class-weighted Cross-Entropy loss based on GTSRB empirical frequencies.
3. **Training & Validation:** Train on disjoint track sequences (`train_split.csv`) and validate on unseen physical sign tracks (`val_split.csv`).
4. **Learning Curves:** Plot training vs validation loss and accuracy across epochs.
5. **Clean Evaluation:** Measure Accuracy, Macro-F1, Precision, and Recall on the official 12,630-image test set.
6. **Error Analysis:** Generate the 43x43 confusion matrix and isolate the most commonly confused sign pairs.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
from src.models import BaselineCNN, count_parameters
from src.dataset import get_data_loaders, DATA_DIR
from src.evaluate import evaluate, plot_and_save_confusion_matrix, identify_most_confused_pairs
from src.utils import CLASS_NAMES, set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active device: {device}")

### 1. Architecture Design & Justification

| Layer Block | Layer Details | Output Shape | Purpose & Justification |
| :--- | :--- | :--- | :--- |
| **Input** | RGB Image | $(3, 32, 32)$ | Standardized resolution without aspect ratio distortion |
| **Block 1** | $2 \times$ [Conv2D(32, $3\times 3$) + BN + ReLU] + MaxPool(2x2) + Dropout(0.2) | $(32, 16, 16)$ | Extracts low-level visual edges, corners, and color gradients |
| **Block 2** | $2 \times$ [Conv2D(64, $3\times 3$) + BN + ReLU] + MaxPool(2x2) + Dropout(0.3) | $(64, 8, 8)$ | Extracts intermediate shapes (triangles, circles, directional lines) |
| **Block 3** | $2 \times$ [Conv2D(128, $3\times 3$) + BN + ReLU] + MaxPool(2x2) + Dropout(0.4) | $(128, 4, 4)$ | Extracts class-specific patterns (speed limit digits, pedestrian icons) |
| **Classifier** | Flatten(2048) $\rightarrow$ Linear(256) + BN + ReLU + Dropout(0.5) $\rightarrow$ Linear(43) | $(43)$ | Dense decision boundary producing class logits |

In [ ]:
model = BaselineCNN(num_classes=43).to(device)
total_params = count_parameters(model)
print(model)
print(f"\nTotal Trainable Parameters: {total_params:,}")

### 2. Training Curves & Convergence
We display the recorded training vs validation loss and accuracy curves across all training epochs.

In [ ]:
import json
history_path = ROOT_DIR / "results" / "baseline_cnn_history.json"

if history_path.exists():
    with open(history_path, "r") as f:
        history = json.load(f)
    
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    axs[0].plot(epochs, history["train_loss"], "o-", label="Train Loss", color="#2b5c8f")
    axs[0].plot(epochs, history["val_loss"], "s-", label="Validation Loss", color="#e07a5f")
    axs[0].set_title("Cross-Entropy Loss Curve")
    axs[0].set_xlabel("Epoch")
    axs[0].set_ylabel("Loss")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)
    
    axs[1].plot(epochs, [a * 100 for a in history["train_acc"]], "o-", label="Train Acc", color="#2b5c8f")
    axs[1].plot(epochs, [a * 100 for a in history["val_acc"]], "s-", label="Val Acc", color="#81b29a")
    axs[1].set_title("Accuracy Curve (%)")
    axs[1].set_xlabel("Epoch")
    axs[1].set_ylabel("Accuracy (%)")
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Training history not found yet. Run src/train.py first.")

### 3. Evaluation on Clean Test Set
We evaluate the best saved checkpoint on the official clean test set of 12,630 images, reporting Accuracy and Macro-F1.

In [ ]:
metrics, top_confused = evaluate(model_name="baseline_cnn")

### 4. Confusion Matrix & Most Confused Sign Pairs
We display the normalized 43x43 confusion matrix and print the top 5 most confused sign pairs.

In [ ]:
cm_img_path = ROOT_DIR / "results" / "confusion_matrices" / "baseline_cnn_cm.png"
if cm_img_path.exists():
    from PIL import Image
    plt.figure(figsize=(14, 14))
    plt.imshow(Image.open(cm_img_path))
    plt.axis("off")
    plt.title("Normalized Confusion Matrix (Baseline CNN)", fontsize=14)
    plt.show()

print("Top Confused Pairs:")
print(top_confused)